# Nén video HEVC 480p (hevc_480p) → Drive `videos/`

Kaggle: bật Internet và T4 GPU. Chạy các cell theo thứ tự. Notebook xử lý từng video độc lập:
tải về, NVDEC giải mã → CUDA resize → NVENC mã hóa H.265 ở độ cao 480 px (giữ tỷ lệ khung hình),
kiểm tra codec/kích thước/thời lượng,
upload vào `videos/` dưới Drive đích, rồi xóa thư mục tạm của video đó.
Cell cuối in thống kê rclone mỗi 5 giây và tiến độ mã hóa theo phần trăm video.

Mặc định chỉ xử lý LHE01. Sửa `START_VIDEO_ID` và `END_VIDEO_ID` nếu muốn chạy nhiều video.
Đặt `OVERWRITE_REMOTE=True` để xử lý lại video đã có ở Drive đích. Lỗi sẽ giữ file local
để có thể chạy lại cell cuối. Audio, nếu có, được mã hóa AAC 128 kb/s.


In [ ]:
from pathlib import Path
import json, os, re, shutil, subprocess

WORK = Path('/kaggle/working/hevc_480p_pipeline')
JOBS_ROOT = WORK / 'jobs'
WORK.mkdir(parents=True, exist_ok=True)
JOBS_ROOT.mkdir(parents=True, exist_ok=True)
START_VIDEO_ID, END_VIDEO_ID = 1, 1
SOURCE_ROOT_FOLDER_ID = '1yPb3EQBPKfKtlP0PGGZHmKDF3pghnRCN'
DEST_ROOT_FOLDER_ID = '11__IfRj1eLK4GZWEeNLs1SQK-DowEU1K'
DEST_FOLDER = 'videos'
RCLONE_CONFIG_SOURCE = Path('/kaggle/input/datasets/vnnoname/rclone-new/rclone.conf')
RCLONE_REMOTE = 'iamtro'
OVERWRITE_REMOTE = False
CQ = 28  # NVENC constant-quality target; lower is higher quality.
PRESET = 'p1'  # Fastest NVENC preset on recent FFmpeg builds.

if not 1 <= START_VIDEO_ID <= END_VIDEO_ID <= 75:
    raise ValueError('Select a range within LHE01...LHE75')
if SOURCE_ROOT_FOLDER_ID == DEST_ROOT_FOLDER_ID:
    raise ValueError('Source and destination Drive roots must differ')
if not 0 <= CQ <= 51 or PRESET not in {'p1','p2','p3','p4','p5','p6','p7'}:
    raise ValueError('Invalid NVENC CQ or preset')

def run(command):
    command = [str(item) for item in command]
    print('$', ' '.join(command[:3]), '...', flush=True)
    subprocess.run(command, check=True)

def run_transfer(command, label):
    command = [str(item) for item in command]
    print(f'{label}: bắt đầu', flush=True)
    command[1:1] = ['--stats', '5s', '--stats-log-level', 'NOTICE']
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(f'{label}: {line.rstrip()}', flush=True)
        if process.wait() != 0:
            raise subprocess.CalledProcessError(process.returncode, command)
    print(f'{label}: hoàn tất', flush=True)

def run_ffmpeg(command, total_seconds):
    command = [str(item) for item in command]
    print('Encode HEVC 480p: bắt đầu', flush=True)
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        progress = {}
        next_percent = 0
        for line in process.stdout:
            line = line.strip()
            if '=' not in line:
                if line:
                    print('FFmpeg:', line, flush=True)
                continue
            key, value = line.split('=', 1)
            progress[key] = value
            if key == 'progress':
                clock = progress.get('out_time', '00:00:00')
                try:
                    hours, minutes, seconds = clock.split(':')
                    elapsed = int(hours) * 3600 + int(minutes) * 60 + float(seconds)
                except ValueError:
                    elapsed = 0
                percent = min(100, int(100 * max(0, elapsed) / total_seconds))
                if percent >= next_percent or value == 'end':
                    print(f'Encode HEVC 480p: {percent}% ({clock}/{total_seconds:.1f}s), '
                          f'speed={progress.get("speed", "?")}, '
                          f'fps={progress.get("fps", "?")}', flush=True)
                    next_percent = percent + 5
                progress.clear()
        if process.wait() != 0:
            raise subprocess.CalledProcessError(process.returncode, command)

def probe(path):
    data = json.loads(subprocess.check_output([
        'ffprobe', '-v', 'error', '-show_entries',
        'stream=index,codec_type,codec_name,width,height:format=duration',
        '-of', 'json', str(path)], text=True))
    videos = [stream for stream in data.get('streams', []) if stream.get('codec_type') == 'video']
    if len(videos) != 1:
        raise ValueError(f'Expected one video stream: {path}')
    duration = float(data.get('format', {}).get('duration', 0))
    if duration <= 0:
        raise ValueError(f'Invalid video duration: {path}')
    return videos[0], duration


## Rclone và danh sách video nguồn

In [ ]:
"""Kaggle rclone installer reused from the embedding notebook; needs WORK and run."""
import os
import shutil
import subprocess
import urllib.request
import zipfile

# downloads.rclone.org có thể trả 403 trên Kaggle. Ưu tiên binary sẵn có,
# sau đó apt của hệ thống; nếu apt không dùng được thì thử release GitHub.
rclone = shutil.which("rclone")
if rclone is None and shutil.which("apt-get") and os.geteuid() == 0:
    apt = shutil.which("apt-get")
    apt_env = {**os.environ, "DEBIAN_FRONTEND": "noninteractive"}
    try:
        install = subprocess.run([apt, "install", "-y", "-qq", "rclone"], env=apt_env,
                                 capture_output=True, text=True, timeout=300)
        if install.returncode != 0:
            subprocess.run([apt, "update", "-qq"], env=apt_env, check=True,
                           capture_output=True, text=True, timeout=300)
            run([apt, "install", "-y", "-qq", "rclone"], env=apt_env)
        rclone = shutil.which("rclone")
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:
        print("apt không cài được rclone:", exc)

if rclone is None:
    release = "v1.75.2"
    archive = f"rclone-{release}-linux-amd64.zip"
    url = f"https://github.com/rclone/rclone/releases/download/{release}/{archive}"
    rclone_zip = WORK / archive
    try:
        request = urllib.request.Request(url, headers={"User-Agent": "kaggle-rclone-setup"})
        with urllib.request.urlopen(request, timeout=120) as src, rclone_zip.open("wb") as dst:
            shutil.copyfileobj(src, dst)
        with zipfile.ZipFile(rclone_zip) as zf:
            matches = [item for item in zf.infolist() if item.filename.endswith("/rclone") and not item.is_dir()]
            if len(matches) != 1:
                raise RuntimeError("Release rclone không chứa đúng một binary")
            binary = WORK / "rclone"
            with zf.open(matches[0]) as src, binary.open("wb") as dst:
                shutil.copyfileobj(src, dst)
        binary.chmod(0o755)
        rclone = str(binary)
    except Exception as exc:
        raise RuntimeError("Không cài được rclone qua apt hoặc GitHub release. "
                           "Hãy gắn rclone binary vào Kaggle và thêm nó vào PATH.") from exc
run([rclone, "version"])

RCLONE_CONFIG = WORK / 'rclone.conf'
if RCLONE_CONFIG_SOURCE.is_file():
    shutil.copyfile(RCLONE_CONFIG_SOURCE, RCLONE_CONFIG)
else:
    secret = os.environ.get('RCLONE_CONFIG')
    if not secret:
        from kaggle_secrets import UserSecretsClient
        secret = UserSecretsClient().get_secret('RCLONE_CONFIG')
    RCLONE_CONFIG.write_text(secret, encoding='utf-8')
    del secret
RCLONE_CONFIG.chmod(0o600)
if shutil.which('ffmpeg') is None or shutil.which('ffprobe') is None:
    raise RuntimeError('This runtime needs ffmpeg and ffprobe')
encoders = subprocess.check_output(['ffmpeg', '-hide_banner', '-encoders'], text=True,
                                   stderr=subprocess.STDOUT)
if not re.search(r'^\s*V[^\n]*\bhevc_nvenc\b', encoders, re.MULTILINE):
    raise RuntimeError('This FFmpeg build does not have the hevc_nvenc encoder')
filters = subprocess.check_output(['ffmpeg', '-hide_banner', '-filters'], text=True,
                                  stderr=subprocess.STDOUT)
if not re.search(r'\bscale_cuda\b', filters):
    raise RuntimeError('This FFmpeg build does not have the scale_cuda filter')
hwaccels = subprocess.check_output(['ffmpeg', '-hide_banner', '-hwaccels'], text=True,
                                   stderr=subprocess.STDOUT)
if 'cuda' not in hwaccels:
    raise RuntimeError('This FFmpeg build does not have CUDA hardware decode')

source_args = [rclone, '--config', RCLONE_CONFIG, '--drive-root-folder-id', SOURCE_ROOT_FOLDER_ID]
dest_args = [rclone, '--config', RCLONE_CONFIG, '--drive-root-folder-id', DEST_ROOT_FOLDER_ID]
remote = RCLONE_REMOTE.rstrip(':') + ':'
entries = json.loads(subprocess.check_output(
    [*map(str, source_args), 'lsjson', remote, '--files-only', '--max-depth', '1'], text=True))
by_name = {}
for entry in entries:
    by_name.setdefault(entry['Name'], []).append(entry)
jobs = []
for number in range(START_VIDEO_ID, END_VIDEO_ID + 1):
    name = f'LHE{number:02d}.mp4'
    matches = by_name.get(name, [])
    if len(matches) != 1 or int(matches[0].get('Size', 0)) <= 0:
        raise ValueError(f'Expected exactly one nonempty source video: {name}')
    jobs.append((name, int(matches[0]['Size'])))
run([*dest_args, 'mkdir', remote + DEST_FOLDER])
existing = set(subprocess.check_output(
    [*map(str, dest_args), 'lsf', remote + DEST_FOLDER, '--files-only'], text=True).splitlines())
print('Selected:', len(jobs), 'videos; already uploaded:', sum(name in existing for name, _ in jobs))


## Mã hóa và upload từng video

In [ ]:
def transcode(source, output):
    partial = output.with_name(output.stem + '.partial.mp4')
    partial.unlink(missing_ok=True)
    try:
        _, source_duration = probe(source)
        run_ffmpeg(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-nostats',
             '-progress', 'pipe:1', '-stats_period', '1', '-y',
             '-hwaccel', 'cuda', '-hwaccel_device', '0', '-hwaccel_output_format', 'cuda',
             '-i', source, '-map', '0:v:0', '-map', '0:a?', '-sn', '-dn',
             '-vf', 'scale_cuda=-2:480:format=nv12',
             '-c:v', 'hevc_nvenc', '-gpu', '0', '-preset', PRESET,
             '-rc', 'vbr', '-cq', CQ, '-b:v', '0', '-tag:v', 'hvc1',
             '-c:a', 'aac', '-b:a', '128k', '-movflags', '+faststart', partial], source_duration)
        video, duration = probe(partial)
        if (video.get('codec_name') != 'hevc' or video.get('height') != 480
                or int(video.get('width', 0)) % 2 or
                abs(duration - source_duration) > max(1.0, source_duration * .01)):
            raise ValueError(f'HEVC 480p output validation failed: {partial}')
        partial.replace(output)
    finally:
        partial.unlink(missing_ok=True)

for index, (name, source_size) in enumerate(jobs, 1):
    if name in existing and not OVERWRITE_REMOTE:
        print(f'[{index}/{len(jobs)}] Skip already uploaded: {name}', flush=True)
        continue
    video_id = Path(name).stem
    job_dir = JOBS_ROOT / video_id
    if job_dir.is_symlink() or job_dir.resolve().parent != JOBS_ROOT.resolve():
        raise ValueError(f'Unsafe job directory: {job_dir}')
    job_dir.mkdir(exist_ok=True)
    source = job_dir / name
    output = job_dir / 'hevc_480p' / name
    output.parent.mkdir(exist_ok=True)
    print(f'[{index}/{len(jobs)}] {name}', flush=True)
    try:
        run_transfer([*source_args, '--checksum', '--retries', '3', 'copyto', remote + name, source],
                     f'Download {name}')
        if not source.is_file() or source.stat().st_size != source_size:
            raise ValueError(f'Source download has wrong size: {source}')
        if not output.is_file():
            transcode(source, output)
        else:
            video, duration = probe(output)
            _, source_duration = probe(source)
            if (video.get('codec_name') != 'hevc' or video.get('height') != 480
                    or abs(duration - source_duration) > max(1.0, source_duration * .01)):
                transcode(source, output)
        run_transfer([*dest_args, '--checksum', '--retries', '3', 'copyto',
                      output, remote + DEST_FOLDER + '/' + name], f'Upload {name}')
        existing.add(name)
        shutil.rmtree(job_dir)
        print(f'Uploaded {DEST_FOLDER}/{name}; local job removed.', flush=True)
    except Exception:
        print(f'Failed {name}; retained local job: {job_dir}', flush=True)
        raise
